# Studio post-fix — Hairball grafo entità filtrato per origin

Ripete l'analisi hairball/componenti connesse di `study_03_graph.ipynb` (union-find su
`entity_links`), ma segmenta i nodi per `origin` del documento sorgente, per verificare
se rimuovendo il contributo di `origin='gdelt'` il grafo si comporta meglio.

**Nota metodologica**: `entity_links` NON ha una colonna `origin` diretta. L'origine di
un link va derivata indirettamente: ogni entità coinvolta in un link è collegata (via
`document_entities`) a uno o più `raw_documents`, ciascuno con un `origin`. Un'entità
può quindi essere "mista" (collegata sia a doc `gdelt` che `rss`). Approccio adottato:
classificare ogni ENTITÀ come `rss_only` (collegata esclusivamente a documenti
`origin='rss'`) o `gdelt_linked` (collegata anche/solo a documenti `origin='gdelt'`),
poi costruire il sottografo indotto sui soli nodi `rss_only` — la lettura più pulita
possibile di "come sarebbe il grafo senza contaminazione GDELT". `source_event` su
`entity_links` non è usato: nel run attuale i link sono quasi tutti `relation_type='co-occurs'`
generati da co-occorrenza documento, non da un singolo evento tracciabile — verificato di
seguito con query diretta.

Riferimento codice: `pathosphere/semantic/graph.py`. Baseline da `study_03_graph.ipynb`:
nodo `GDELT` grado 3962/89838 archi (4.4%), componente gigante 9666/10192 nodi (94.8%).

In [1]:
import sys, struct, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pathosphere").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from pathosphere.db.schema import get_connection

DB_PATH = REPO_ROOT / "data" / "db" / "pathosphere.db"
conn = get_connection(DB_PATH)
print(f"DB: {DB_PATH}  exists={DB_PATH.exists()}")

def q(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

q("SELECT COUNT(*) AS raw_documents FROM raw_documents")


DB: /Users/dom/Documents/GitHub/pathosphere/data/db/pathosphere.db  exists=True


,raw_documents
0,180505


## 0. Verifica preliminare: `source_event` è popolato su `entity_links`?

In [2]:
source_event_cov = q("""
SELECT relation_type,
       COUNT(*) AS n,
       SUM(CASE WHEN source_event IS NOT NULL THEN 1 ELSE 0 END) AS con_source_event
FROM entity_links GROUP BY relation_type
""")
print("Se source_event fosse popolato potremmo derivare origin via events.origin. "
      "Verifichiamo:")
source_event_cov


Se source_event fosse popolato potremmo derivare origin via events.origin. Verifichiamo:


,relation_type,n,con_source_event
0,co-occurs,123047,0


`source_event` risulta non utilizzabile per derivare `origin` in modo affidabile
(vedi output sopra) — si procede con l'approccio via `document_entities` descritto in
apertura.

## 1. Baseline grafo intero (replica study_03, dati freschi)

In [3]:
links_summary = q("""
SELECT relation_type, COUNT(*) AS n, AVG(strength) AS avg_strength
FROM entity_links GROUP BY relation_type
""")
links_summary


,relation_type,n,avg_strength
0,co-occurs,123047,0.139456


In [4]:
edges = q("SELECT entity_a, entity_b, strength FROM entity_links")
parent = {}

def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    rx, ry = find(x), find(y)
    if rx != ry:
        parent[rx] = ry

for _, r in edges.iterrows():
    union(int(r.entity_a), int(r.entity_b))

roots = Counter(find(x) for x in parent)
comp_sizes_full = pd.Series(sorted(roots.values(), reverse=True))
n_nodes_full = len(parent)
n_edges_full = len(edges)
giant_full = int(comp_sizes_full.iloc[0]) if len(comp_sizes_full) else 0
pct_giant_full = giant_full / n_nodes_full * 100 if n_nodes_full else 0

print(f"nodi coinvolti in almeno 1 link: {n_nodes_full}")
print(f"archi totali: {n_edges_full}")
print(f"numero componenti connesse: {len(roots)}")
print(f"componente gigante: {giant_full}/{n_nodes_full} nodi ({pct_giant_full:.1f}%)")


nodi coinvolti in almeno 1 link: 13278
archi totali: 123047
numero componenti connesse: 149
componente gigante: 12664/13278 nodi (95.4%)


In [5]:
deg = q("""
SELECT entity_id, COUNT(*) AS degree FROM (
  SELECT entity_a AS entity_id FROM entity_links
  UNION ALL
  SELECT entity_b AS entity_id FROM entity_links
) GROUP BY entity_id ORDER BY degree DESC
""")
entities_all = q("SELECT id, name, entity_type FROM entities")
top_degree = deg.head(20).merge(entities_all, left_on="entity_id", right_on="id")
top_degree[["name", "entity_type", "degree"]]


,name,entity_type,degree
0,GDELT,company,3962
1,US,location,1737
2,POLICE,company,922
3,span><strong,other,892
4,China,location,798
5,Iran,location,788
6,Russia,location,733
7,India,location,721
8,Trump,person,706
9,Donald Trump,person,656


In [6]:
gdelt_row = top_degree[top_degree["name"] == "GDELT"]
total_edges_x2 = int(deg["degree"].sum())  # ogni arco conta 2 volte (entity_a + entity_b)

BASELINE_GDELT_DEGREE = 3962
BASELINE_TOTAL_EDGES = 89838
BASELINE_GIANT = 9666
BASELINE_TOTAL_NODES = 10192
BASELINE_GIANT_PCT = 94.8

if len(gdelt_row):
    gd_today = int(gdelt_row.iloc[0]["degree"])
    print(f"Grado nodo 'GDELT' OGGI: {gd_today} su {n_edges_full} archi totali "
          f"({gd_today / n_edges_full * 100:.1f}%)")
    print(f"Grado nodo 'GDELT' in study_03 (baseline pre-fix): {BASELINE_GDELT_DEGREE} su "
          f"{BASELINE_TOTAL_EDGES} archi ({BASELINE_GDELT_DEGREE / BASELINE_TOTAL_EDGES * 100:.1f}%)")
    delta = gd_today - BASELINE_GDELT_DEGREE
    trend = "aumentato" if delta > 0 else "diminuito" if delta < 0 else "invariato"
    print(f"Delta: {delta:+d} ({trend})")
else:
    gd_today = None
    print("'GDELT' non è tra i top-20 per grado in questo run — sarebbe un netto miglioramento "
          f"rispetto alla baseline (grado {BASELINE_GDELT_DEGREE} in study_03).")


Grado nodo 'GDELT' OGGI: 3962 su 123047 archi totali (3.2%)
Grado nodo 'GDELT' in study_03 (baseline pre-fix): 3962 su 89838 archi (4.4%)
Delta: +0 (invariato)


## 2. Classificazione entità per origin (rss_only vs gdelt_linked)

In [7]:
entity_origin = q("""
SELECT de.entity_id, rd.origin
FROM document_entities de
JOIN raw_documents rd ON rd.id = de.document_id
GROUP BY de.entity_id, rd.origin
""")
origins_per_entity = entity_origin.groupby("entity_id")["origin"].apply(set)

rss_only_ids = set(origins_per_entity[origins_per_entity == {"rss"}].index)
gdelt_linked_ids = set(entity_origin.loc[entity_origin["origin"] == "gdelt", "entity_id"])

print(f"entità collegate SOLO a documenti origin=rss: {len(rss_only_ids)}")
print(f"entità collegate (anche) a documenti origin=gdelt: {len(gdelt_linked_ids)}")
print(f"entità totali con almeno un document_entities: {len(origins_per_entity)}")


entità collegate SOLO a documenti origin=rss: 10473
entità collegate (anche) a documenti origin=gdelt: 3967
entità totali con almeno un document_entities: 14494


## 3. Sottografo rss-only — componente connessa più grande

In [8]:
# ricostruzione union-find ristretta ai soli link dove ENTRAMBI i nodi sono rss_only
parent_rss = {}

def find_rss(x):
    parent_rss.setdefault(x, x)
    while parent_rss[x] != x:
        parent_rss[x] = parent_rss[parent_rss[x]]
        x = parent_rss[x]
    return x

def union_rss(x, y):
    rx, ry = find_rss(x), find_rss(y)
    if rx != ry:
        parent_rss[rx] = ry

n_edges_rss_only = 0
for _, r in edges.iterrows():
    a, b = int(r.entity_a), int(r.entity_b)
    if a in rss_only_ids and b in rss_only_ids:
        union_rss(a, b)
        n_edges_rss_only += 1

roots_rss = Counter(find_rss(x) for x in parent_rss)
comp_sizes_rss = pd.Series(sorted(roots_rss.values(), reverse=True))
n_nodes_rss = len(parent_rss)
giant_rss = int(comp_sizes_rss.iloc[0]) if len(comp_sizes_rss) else 0
pct_giant_rss = giant_rss / n_nodes_rss * 100 if n_nodes_rss else 0

print(f"archi rss-only (entrambi i nodi rss_only): {n_edges_rss_only}")
print(f"nodi coinvolti nel sottografo rss-only: {n_nodes_rss}")
print(f"numero componenti connesse (rss-only): {len(roots_rss)}")
print(f"componente gigante rss-only: {giant_rss}/{n_nodes_rss} nodi ({pct_giant_rss:.1f}%)" if n_nodes_rss else "sottografo vuoto")


archi rss-only (entrambi i nodi rss_only): 90369
nodi coinvolti nel sottografo rss-only: 9283
numero componenti connesse (rss-only): 180
componente gigante rss-only: 8573/9283 nodi (92.4%)


In [9]:
comp_sizes_rss.head(10) if len(comp_sizes_rss) else "nessuna componente"


0    8573
1      66
2      13
3      11
4      10
5       9
6       9
7       9
8       9
9       9
dtype: int64

## 4. Top grado nel sottografo rss-only

In [10]:
deg_rss = q("""
SELECT entity_a AS entity_id FROM entity_links
UNION ALL
SELECT entity_b AS entity_id FROM entity_links
""")
deg_rss = deg_rss[deg_rss["entity_id"].isin(rss_only_ids)]
deg_rss_counts = deg_rss["entity_id"].value_counts().reset_index()
deg_rss_counts.columns = ["entity_id", "degree"]
top_rss = deg_rss_counts.head(20).merge(entities_all, left_on="entity_id", right_on="id")
top_rss[["name", "entity_type", "degree"]]


,name,entity_type,degree
0,span><strong,other,892
1,Iran,location,788
2,Russia,location,733
3,India,location,721
4,Trump,person,706
5,Donald Trump,person,656
6,Pakistan,location,644
7,Ukraine,location,597
8,Russian,other,566
9,Chinese,other,503


## 5. Confronto quantitativo esplicito — prima (study_02/03) vs dopo (questo notebook)

In [11]:
print("=" * 70)
print("CONFRONTO PRIMA (study_02/study_03, contaminato) vs DOPO (post-fix, questo notebook)")
print("=" * 70)
print()
print(f"{'Metrica':<45}{'PRIMA':>12}{'DOPO':>15}")
print("-" * 72)
print(f"{'Grado nodo GDELT':<45}{BASELINE_GDELT_DEGREE:>12}{(gd_today if gd_today is not None else 0):>15}")
print(f"{'Archi totali nel grafo':<45}{BASELINE_TOTAL_EDGES:>12}{n_edges_full:>15}")
print(f"{'Nodi totali nel grafo':<45}{BASELINE_TOTAL_NODES:>12}{n_nodes_full:>15}")
print(f"{'Componente gigante (nodi)':<45}{BASELINE_GIANT:>12}{giant_full:>15}")
print(f"{'Componente gigante (% nodi)':<45}{BASELINE_GIANT_PCT:>11.1f}%{pct_giant_full:>14.1f}%")
print()
rss_pct_str = f"{pct_giant_rss:.1f}%"
print(f"{'Sottografo rss-only (senza nodi gdelt-linked)':<45}{'n/a':>12}{rss_pct_str:>15}")
print(f"  nodi rss-only: {n_nodes_rss} | componente gigante rss-only: {giant_rss} ({pct_giant_rss:.1f}%)")
print()

miglioramento_pct_punti = BASELINE_GIANT_PCT - pct_giant_rss
print(f"Il sottografo rss-only riduce la quota in componente gigante di "
      f"{miglioramento_pct_punti:.1f} punti percentuali rispetto alla baseline contaminata "
      f"({BASELINE_GIANT_PCT:.1f}% -> {pct_giant_rss:.1f}%)." if n_nodes_rss else
      "Sottografo rss-only vuoto: nessun confronto possibile.")


CONFRONTO PRIMA (study_02/study_03, contaminato) vs DOPO (post-fix, questo notebook)

Metrica                                             PRIMA           DOPO
------------------------------------------------------------------------
Grado nodo GDELT                                     3962           3962
Archi totali nel grafo                              89838         123047
Nodi totali nel grafo                               10192          13278
Componente gigante (nodi)                            9666          12664
Componente gigante (% nodi)                         94.8%          95.4%

Sottografo rss-only (senza nodi gdelt-linked)         n/a          92.4%
  nodi rss-only: 9283 | componente gigante rss-only: 8573 (92.4%)

Il sottografo rss-only riduce la quota in componente gigante di 2.4 punti percentuali rispetto alla baseline contaminata (94.8% -> 92.4%).


## Sintesi criticità osservate in questo run

In [12]:
print("--- Sintesi (valori da questo run) ---")
if gd_today is not None:
    print(f"1. Nodo 'GDELT' ancora presente e ancora top-degree: {gd_today}/{n_edges_full} archi "
          f"({gd_today/n_edges_full*100:.1f}%) vs baseline {BASELINE_GDELT_DEGREE}/{BASELINE_TOTAL_EDGES} "
          f"({BASELINE_GDELT_DEGREE/BASELINE_TOTAL_EDGES*100:.1f}%) — il fix gdelt_anomaly non ha "
          "toccato il grafo esistente (entity_links storici restano intatti, nessun cleanup retroattivo).")
else:
    print("1. Nodo 'GDELT' non più tra i top-20 per grado — miglioramento netto.")
print(f"2. Componente gigante grafo INTERO oggi: {giant_full}/{n_nodes_full} nodi ({pct_giant_full:.1f}%) "
      f"vs baseline study_03 {BASELINE_GIANT}/{BASELINE_TOTAL_NODES} nodi ({BASELINE_GIANT_PCT:.1f}%) — "
      f"{'sostanzialmente invariata' if abs(pct_giant_full - BASELINE_GIANT_PCT) < 2 else 'cambiata'}: "
      "il fix agisce solo su NUOVI eventi (gdelt_anomaly bypassa NER/graph interamente), non pulisce "
      "gli entity_links generati prima del fix.")
print(f"3. Sottografo rss-only (rimuovendo nodi contaminati da gdelt): componente gigante "
      f"{giant_rss}/{n_nodes_rss} nodi ({pct_giant_rss:.1f}%) — "
      f"{'migliora' if pct_giant_rss < BASELINE_GIANT_PCT else 'non migliora'} rispetto alla baseline "
      f"{BASELINE_GIANT_PCT:.1f}%, indicando che l'hairball non è dovuto SOLO al nodo GDELT ma anche "
      "a entità generiche ALL CAPS (POLICE, GOVERNMENT, ecc.) che collegano comunque tra loro anche "
      "senza il nodo GDELT esplicito.")
print("4. CONCLUSIONE: il fix gdelt_anomaly.py prevendrà la contaminazione FUTURA del grafo (nuove "
      "anomalie promosse direttamente a events senza passare da NER/graph), ma il grafo ESISTENTE "
      "resta contaminato fino a un cleanup esplicito di entities/document_entities/entity_links "
      "per i documenti origin=gdelt processati pre-fix.")


--- Sintesi (valori da questo run) ---
1. Nodo 'GDELT' ancora presente e ancora top-degree: 3962/123047 archi (3.2%) vs baseline 3962/89838 (4.4%) — il fix gdelt_anomaly non ha toccato il grafo esistente (entity_links storici restano intatti, nessun cleanup retroattivo).
2. Componente gigante grafo INTERO oggi: 12664/13278 nodi (95.4%) vs baseline study_03 9666/10192 nodi (94.8%) — sostanzialmente invariata: il fix agisce solo su NUOVI eventi (gdelt_anomaly bypassa NER/graph interamente), non pulisce gli entity_links generati prima del fix.
3. Sottografo rss-only (rimuovendo nodi contaminati da gdelt): componente gigante 8573/9283 nodi (92.4%) — migliora rispetto alla baseline 94.8%, indicando che l'hairball non è dovuto SOLO al nodo GDELT ma anche a entità generiche ALL CAPS (POLICE, GOVERNMENT, ecc.) che collegano comunque tra loro anche senza il nodo GDELT esplicito.
4. CONCLUSIONE: il fix gdelt_anomaly.py prevendrà la contaminazione FUTURA del grafo (nuove anomalie promosse diretta